<img src="../assets/tumor_twin.png" alt="Tumor Twin" width="500"/>

# Coupled PDE demo: HGG patient data + immune–tumor model

This notebook follows **the same data path and preprocessing as [HGG_Demo](HGG_Demo.ipynb)** (`HGG_demo_001`: JSON + NIfTI under `input_files/HGG_demo_001/`). The only modeling change is **`ImmuneResponse3D`** instead of `ReactionDiffusion3D`, so the ODE state is stacked as `(C, D, H, W)` (tumor + lymphocytes).

**Requirements:** Place the imaging archive for `HGG_demo_001` next to the JSON (same layout as the main HGG tutorial). If files are missing, `HGGPatientData.from_file` will fail—the error is the same as in **HGG_Demo**.

**Takeaway:** After `solver.solve`, use `trajectory_to_map_list(..., 0)` for tumor-only maps so `plot_predicted_TCC`, `plot_cellularity_map`, and LM-style residuals match the single-field tutorial.

---
## Table of contents
- [Imports & paths](#imports--paths)
- [Load patient & ADC-derived cellularity](#load-patient--adc-derived-cellularity)
- [Treatment specs & immune model](#treatment-specs--immune-model)
- [Forward solve](#forward-solve)
- [Postprocess (tumor channel)](#postprocess-tumor-channel)
- [Residuals at visit times](#residuals-at-visit-times)
---


In [1]:
# Imports: same stack as HGG_Demo, plus ImmuneResponse3D and pde_workflow
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
from datetime import timedelta
from pathlib import Path

from pydantic import FilePath

from tumortwin.models.immune_3d import ImmuneResponse3D
from tumortwin.pde_workflow import (
    fields_at_times_from_trajectory,
    initial_pde_state_from_tumor_field,
    select_timepoint_indices,
    spatiotemporal_residual_vector,
    squared_error_loss,
    trajectory_to_map_list,
)
from tumortwin.postprocessing import (
    plot_cellularity_map,
    plot_imaging_summary,
    plot_patient_timeline,
    plot_predicted_TCC,
)
from tumortwin.preprocessing import ADC_to_cellularity, compute_carrying_capacity
from tumortwin.solvers import TorchDiffEqSolver, TorchDiffEqSolverOptions
from tumortwin.types import (
    ChemotherapySpecification,
    CropSettings,
    CropTarget,
    RadiotherapySpecification,
)
from tumortwin.types.hgg_data import HGGPatientData
from tumortwin.utils import daterange, days_since_first

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

%matplotlib inline
matplotlib.rc("font", weight="normal", size=10)
matplotlib.rc("figure", dpi=300)
matplotlib.rc("savefig", dpi=300)


ModuleNotFoundError: No module named 'tumortwin'

In [ ]:
# Same paths as HGG_Demo (run this notebook from `tutorials/`)
data_path = Path("../input_files/HGG_demo_001")
PATIENT_INFO_PATH = FilePath(str(data_path / "HGG_demo_001.json"))
IMAGE_PATH = FilePath(str(data_path))
crop_settings = CropSettings(crop_to=CropTarget.ROI_ENHANCE, padding=10, visit_index=-1)

patient_data = HGGPatientData.from_file(
    PATIENT_INFO_PATH, image_dir=IMAGE_PATH, crop_settings=crop_settings
)
patient_data


In [ ]:
plot_patient_timeline(patient_data)
plt.show()
plot_imaging_summary(patient_data)
plt.show()


In [ ]:
measured_cellularity_maps = [
    ADC_to_cellularity(
        visit.adc_image, visit.roi_enhance_image, visit.roi_nonenhance_image
    )
    for visit in patient_data.visits
]
carrying_capacity = compute_carrying_capacity(patient_data.brainmask_image)
print("Number of visits:", len(measured_cellularity_maps), "carrying_capacity:", carrying_capacity)


In [ ]:
# Identical radiotherapy / chemotherapy specifications as HGG_Demo
rt = RadiotherapySpecification(
    alpha=0.025,
    alpha_beta_ratio=10,
    times=[r.time for r in patient_data.radiotherapy],
    doses=[r.dose for r in patient_data.radiotherapy],
)
ct = ChemotherapySpecification(
    sensitivity=0.5,
    decay_rate=9.2420,
    times=[c.time for c in patient_data.chemotherapy],
    doses=[c.dose for c in patient_data.chemotherapy],
)


In [ ]:
# Immune–tumor parameters (order of magnitude similar to HGG k, D); tune for your study
D1 = torch.tensor(0.025, requires_grad=True, device=device)
mu1 = torch.tensor(0.05, requires_grad=True, device=device)
gamma12 = torch.tensor(0.02, requires_grad=True, device=device)
D4 = torch.tensor(0.025, requires_grad=True, device=device)
gamma21 = torch.tensor(0.02, requires_grad=True, device=device)
v = [0.0, 0.0, 0.0]

initial_tumor = torch.from_numpy(measured_cellularity_maps[0].array).float().to(device)

model = ImmuneResponse3D(
    D1=D1,
    mu1=mu1,
    gamma12=gamma12,
    D4=D4,
    gamma21=gamma21,
    v=v,
    patient_data=patient_data,
    initial_time=patient_data.visits[0].time,
    initial_u1=initial_tumor,
    radiotherapy_specification=rt,
    chemotherapy_specifications=[ct],
    require_grad=True,
    device=device,
)

# Stacked IC (2, D, H, W): must match what the solver integrates
u0 = model.get_initial_state()
u_alt = initial_pde_state_from_tumor_field(
    initial_tumor,
    num_components=model.num_state_components,
    other_fill=float(model.u4_source.detach().cpu().item()),
    device=device,
)
torch.testing.assert_close(u0, u_alt)
print("Initial state shape:", tuple(u0.shape))


In [ ]:
solver_options = TorchDiffEqSolverOptions(
    step_size=timedelta(days=0.5),
    use_adjoint=True,
    device=device,
    method="rk4",
)
solver = TorchDiffEqSolver(model, solver_options)


In [ ]:
timepoints = daterange(
    patient_data.visits[0].time, patient_data.visits[-1].time, timedelta(days=0.5)
)
_, trajectory = solver.solve(timepoints=timepoints, u_initial=u0)
print("Trajectory shape (T, C, D, H, W):", tuple(trajectory.shape))


In [ ]:
# Tumor component only — same as feeding HGG_Demo predictions into TCC / maps
tumor_maps = trajectory_to_map_list(trajectory, component_idx=0)

fig, ax = plt.subplots(1, 1, figsize=(5, 2.5))
plot_predicted_TCC(tumor_maps, timepoints, ax=ax, carrying_capacity=carrying_capacity)
ax.set_title("Predicted TCC (tumor field, component 0)")
plt.tight_layout()
plt.show()

# Compare a few visits (subset like HGG_Demo)
visit_slice = patient_data.visit_days[::2]
n = len(visit_slice)
fig, axes = plt.subplots(2, n, figsize=(3 * n, 5), squeeze=False)
t_tensor = torch.tensor(
    [days_since_first(t, timepoints[0]) for t in timepoints],
    dtype=torch.float32,
    device=trajectory.device,
)
idx_vis = select_timepoint_indices(t_tensor, visit_slice, atol=0.51)
for i, vd in enumerate(visit_slice):
    t_idx = idx_vis[i]
    plot_cellularity_map(
        tumor_maps[t_idx].cpu(), patient_data, time=vd, ax=axes[0, i]
    )
    plot_cellularity_map(
        torch.tensor(measured_cellularity_maps[2 * i].array).float(),
        patient_data,
        time=vd,
        ax=axes[1, i],
    )
axes[0, 0].set_ylabel("Predicted (immune model)")
axes[1, 0].set_ylabel("Measured (ADC)")
plt.tight_layout()
plt.show()


In [ ]:
# LM-ready residual on tumor maps at early visits (same spirit as HGG_Demo Step 7)
n_visits_cal = min(4, len(patient_data.visits))
visit_days_cal = patient_data.visit_days[:n_visits_cal]
idx_cal = select_timepoint_indices(t_tensor, visit_days_cal, atol=0.51)
pred_maps = fields_at_times_from_trajectory(trajectory, idx_cal, component_idx=0)
meas_maps = [
    torch.tensor(measured_cellularity_maps[j].array, dtype=torch.float32, device=device)
    for j in range(n_visits_cal)
]
res = spatiotemporal_residual_vector(pred_maps, meas_maps)
loss = squared_error_loss(res)
print("Residual length:", res.numel(), "SSE:", float(loss.detach().cpu()))

# Full spatiotemporal SSE backprop through torchdiffeq is expensive on the full HGG grid.
# For gradients, follow HGG_Demo Step 5 / HGG_Gradients: define a scalar QoI (e.g. TCC at one visit)
# on `tumor_maps[t_idx]` and call `.backward()` on that scalar.


## Summary

- **Data** and preprocessing match **HGG_Demo** (`HGG_demo_001`, ADC cellularity, ROI crop, RT/CT from JSON).
- **Model:** `ImmuneResponse3D` with `u0 = model.get_initial_state()` (`(2, D, H, W)`).
- **Plots / TCC:** `trajectory_to_map_list(trajectory, 0)` feeds the same plotting functions as the scalar model.
- **Calibration:** flatten `res` with `spatiotemporal_residual_vector` and pass a parameterised forward map into `LMoptimizer` as in **HGG_Demo** Step 7.
